In [7]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.mediation import Mediation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
print(plt.rcParams["font.family"])
plt.rcParams["font.sans-serif"] = ["Arial"] + plt.rcParams["font.sans-serif"]

['sans-serif']


In [18]:
# Load formatted cytokines data
df = pd.read_csv("../data/CV_plasma_Combined_formatted_20260731.csv")
df

,sample_names,sample_id,cohort,egf,ifnalpha2,ifngamma,il1alpha,il1beta,il4,il6,...,globalhealthstatusql2,qlq30score,hn35,pathologicstage,lvi,pni,survival_status,sx_to_lastfu,survival_2yrs,ss
0,FF301C,FF301C,LLU,29.45,25.24,90.00,6.40,3.95,4.21,3.54,...,75.00,81.62,30.56,stage_I,no,no,alive,1839,alive,1
1,FF302C,FF302C,LLU,71.50,9.93,46.35,0.00,0.00,2.33,3.42,...,16.67,46.11,47.78,stage_IVA,no,yes,alive,1721,alive,4
2,FF303C,FF303C,LLU,54.54,18.38,54.01,46.49,18.90,1.37,4.15,...,50.00,57.56,45.68,stage_IVA,yes,no,death,48,death,4
3,FF304C,FF304C,LLU,132.67,27.47,81.82,6.89,3.95,4.56,9.69,...,0.00,19.23,94.44,stage_IVA,no,yes,death,131,death,3
4,FF305C,FF305C,LLU,28.01,13.63,66.20,5.44,0.92,3.04,2.42,...,75.00,76.67,17.44,stage_II,no,no,alive,1679,alive,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,76,S0076,UAB,61.90,122.36,103.74,552.31,15.62,5864.00,199.75,...,NaN,NaN,NaN,stage_IVB,yes,yes,death,882,alive,3
208,77,S0077,UAB,18.86,91.86,71.49,22.67,5.42,146.71,18.96,...,NaN,NaN,NaN,stage_II,no,no,alive,1273,alive,1
209,78,S0078,UAB,12.99,66.46,51.61,9.46,2.89,28.61,2.28,...,NaN,NaN,NaN,stage_III,yes,yes,alive,1250,alive,4
210,79,S0079,UAB,13.66,75.02,40.79,8.96,3.81,38.38,5.04,...,NaN,NaN,NaN,stage_II,no,no,alive,1183,alive,2


In [59]:
# Encode survival status as binary variable
df["survival_status_encoded"] = df["survival_status"].map({"alive": 0, "death": 1})
df["lvi_encoded"] = df["lvi"].map({"no": 0, "yes": 1})
df["pni_encoded"] = df["pni"].map({"no": 0, "yes": 1})

In [64]:
sc = StandardScaler()

cols = ["il10", "tnfalpha", "ss"]

df_standardization_df = df.copy()
df_standardization_df[cols] = (
    df_standardization_df.groupby("cohort")[cols]
    .transform(lambda x: sc.fit_transform(x.to_numpy().reshape(-1, 1)).ravel())
)

df_standardization_df.loc[:, ["il10", "tnfalpha", "ss"]]

,il10,tnfalpha,ss
0,-0.485307,0.004131,-1.501561
1,0.215519,0.635905,0.711266
2,0.442933,0.343245,0.711266
3,-0.321014,0.881182,-0.026343
4,-0.419334,1.943120,-1.501561
...,...,...,...
207,4.921763,0.721482,0.355378
208,1.954145,1.710124,-1.156869
209,-0.602483,-0.167541,1.111501
210,-0.430010,0.267160,-0.400745


In [63]:
mediator = "ss"
exposure = "tnfalpha"

# Mediator model
mediator_model = smf.ols(
    formula=f"{mediator} ~ {exposure}",
    # family=sm.families.Binomial(),
    data=df_standardization_df
)

# Outcome model
outcome_model = smf.phreg(
    formula=f"sx_to_lastfu ~ {exposure} + {mediator}",
    status=df_standardization_df["survival_status_encoded"],
    data=df_standardization_df
)

med = Mediation(
    outcome_model,
    mediator_model,
    exposure=exposure,
    mediator=mediator,
    outcome_predict_kwargs={"pred_only": True},
)

res = med.fit(n_rep=5000)
print(res.summary())

                          Estimate  Lower CI bound  Upper CI bound  P-value
ACME (control)            0.054150       -0.128378        0.242710   0.5528
ACME (treated)            0.054150       -0.128378        0.242710   0.5528
ADE (control)             0.212484        0.029315        0.394586   0.0244
ADE (treated)             0.212484        0.029315        0.394586   0.0244
Total effect              0.266633        0.009399        0.527847   0.0420
Prop. mediated (control)  0.210186       -1.574512        0.897104   0.5180
Prop. mediated (treated)  0.210186       -1.574512        0.897104   0.5180
ACME (average)            0.054150       -0.128378        0.242710   0.5528
ADE (average)             0.212484        0.029315        0.394586   0.0244
Prop. mediated (average)  0.210186       -1.574512        0.897104   0.5180
